# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The final ranked queue, combining the validated model (Weeks 5–6) with the transparent baseline's reason codes (Week 4). The model is **retrained on the full Lane 4 slice** for this production ranking — Weeks 5–6 already established, on a proper client-holdout split, that it beats the baseline; this pass uses all available data rather than holding a quarter of it back, since the honest performance estimate was already earned separately.

In [1]:
import pandas as pd
import numpy as np
import os
import json

pd.set_option('display.width', 160)

df = pd.read_csv('/content/content_refresh_anonymized.csv')
filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')
lane4 = filtered[(filtered['avg_position'] > 0) & (filtered['avg_position'] <= 20)
                  & (filtered['impressions_90d'] >= 500)].copy()

lane4['expected_ctr_for_tier'] = lane4.groupby('position_tier')['ctr'].transform('median')
lane4['ctr_gap_score'] = (lane4['expected_ctr_for_tier'] - lane4['ctr']).clip(lower=0)
median_scroll = lane4['scroll_rate'].median()
lane4['engagement_deficit'] = ((lane4['engagement_rate'] == 0) & (lane4['scroll_rate'] < median_scroll)).astype(int)
lane4['log_impressions'] = np.log1p(lane4['impressions_90d'])
lane4['word_count_missing'] = lane4['word_count'].isna().astype(int)
lane4['word_count_filled'] = lane4['word_count'].fillna(-1)

cat_cols = ['position_tier', 'competition_level', 'main_intent']
num_cols = ['avg_position', 'log_impressions', 'ctr', 'content_age_days', 'word_count_filled', 'word_count_missing']
dummies = pd.get_dummies(lane4[cat_cols], columns=cat_cols)
lane4_enc = pd.concat([lane4[num_cols].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
# NOTE: 'ctr' is intentionally NOT duplicated into `extra` -- it's already present via num_cols
# above as a real model feature. Adding it again under the same name would create a duplicate
# column, and excluding-by-name against `extra.columns` would then silently strip BOTH copies
# from feature_cols -- quietly dropping ctr as a feature with no error at all.
# NOTE: content_age_days is also NOT duplicated here -- same reasoning as ctr above,
# it's already present via num_cols. Both ctr and content_age_days now travel correctly
# through the sort as genuine feature columns already inside lane4_enc/ranked.
extra = lane4[['content_id', 'client_id', 'position_tier', 'impressions_90d',
               'ctr_gap_score', 'engagement_deficit']].reset_index(drop=True)
lane4_enc = pd.concat([lane4_enc, extra], axis=1)
feature_cols = [c for c in lane4_enc.columns if c not in extra.columns]
assert 'ctr' in feature_cols, "ctr must remain a feature -- check for column-name collisions"
assert not lane4_enc.columns.duplicated().any(), "no duplicate columns allowed in lane4_enc"

from sklearn.ensemble import RandomForestClassifier

final_model = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42,
                                      n_jobs=-1, class_weight='balanced')
final_model.fit(lane4_enc[feature_cols], lane4_enc['engagement_deficit'])
lane4_enc['pred_prob'] = final_model.predict_proba(lane4_enc[feature_cols])[:, 1]

lane4_enc['reason_low_ctr_vs_tier'] = lane4_enc['ctr_gap_score'] > 0
lane4_enc['reason_high_stakes'] = lane4_enc['impressions_90d'] >= lane4_enc['impressions_90d'].quantile(0.75)
lane4_enc['reason_model_flagged'] = lane4_enc['pred_prob'] >= 0.5

prob_80th = lane4_enc['pred_prob'].quantile(0.80)
lane4_enc['high_confidence'] = (lane4_enc['pred_prob'] >= prob_80th) & (lane4_enc['impressions_90d'] >= 500)

ranked = lane4_enc.sort_values(['pred_prob', 'ctr_gap_score'], ascending=[False, False]).reset_index(drop=True)
ranked['rank'] = ranked.index + 1

print(f'ranked queue: {len(ranked)} pages')
print(f'high_confidence pages (top quintile by model probability AND impressions>=500): {ranked["high_confidence"].sum()}')
print()
review_cols = ['rank', 'content_id', 'position_tier', 'impressions_90d', 'ctr', 'pred_prob',
               'reason_low_ctr_vs_tier', 'reason_high_stakes', 'reason_model_flagged', 'high_confidence']
ranked[review_cols].head(10)

ranked queue: 12023 pages
high_confidence pages (top quintile by model probability AND impressions>=500): 2405



,rank,content_id,position_tier,impressions_90d,ctr,pred_prob,reason_low_ctr_vs_tier,reason_high_stakes,reason_model_flagged,high_confidence
0,1,content_ae60399f8455,striking,529,0.0,0.880068,True,False,True,True
1,2,content_c6e59d4bfbfb,striking,518,0.0,0.879607,True,False,True,True
2,3,content_02866c352877,striking,524,0.0,0.878697,True,False,True,True
3,4,content_1c9181753d4b,striking,525,0.0,0.876269,True,False,True,True
4,5,content_d48a0d388685,page_1,902,0.0,0.872461,True,False,True,True
5,6,content_89075a9f4b1e,striking,551,0.0,0.871765,True,False,True,True
6,7,content_08dc2f4d70d3,striking,564,0.0,0.869545,True,False,True,True
7,8,content_829eba7c571f,striking,500,0.0,0.868900,True,False,True,True
8,9,content_822f03538863,striking,522,0.0,0.867882,True,False,True,True
9,10,content_420535f56d33,striking,1249,0.0,0.866196,True,False,True,True


**In words a human trusts:** pages sit at the top because the model's own risk score puts them there, and every one carries at least the `low_ctr_vs_tier` reason (their CTR sits below what similar-ranked pages typically earn). `high_stakes` marks pages where getting the call wrong costs more attention either way (top-quartile impression volume), and `high_confidence` marks the subset where the model's score is unusually high *and* there's enough traffic to trust the number.

**Archetype → action mapping.** Rather than one blended score, every page falls into exactly one of four archetypes, built from the two *independent* reason signals already computed (CTR gap vs. tier, and the separate engagement-deficit check) — each archetype gets a distinct recommended action, not a generic "review this."

In [2]:
def archetype(row):
    if row['reason_low_ctr_vs_tier'] and row['engagement_deficit']:
        return 'dual_underperformer'
    elif row['reason_low_ctr_vs_tier']:
        return 'ctr_only_underperformer'
    elif row['engagement_deficit']:
        return 'engagement_only_underperformer'
    else:
        return 'healthy'

ranked['archetype'] = ranked.apply(archetype, axis=1)

action_map = {
    'ctr_only_underperformer':        'Rewrite title/meta description -- position and content are fine, the listing is not earning its clicks.',
    'engagement_only_underperformer': 'Review on-page content/layout -- clicks are fine, but visitors are not engaging once they arrive.',
    'dual_underperformer':            'Full review, highest priority -- both entry (CTR) and on-page (engagement) signals are weak together.',
    'healthy':                        'Monitor only -- no action recommended from this queue.',
}

counts = ranked['archetype'].value_counts()
pct = (ranked['archetype'].value_counts(normalize=True) * 100).round(1)
archetype_table = pd.DataFrame({'count': counts, 'pct_of_queue': pct, 'recommended_action': [action_map[a] for a in counts.index]})
print(archetype_table)

                                count  pct_of_queue                                 recommended_action
archetype                                                                                             
healthy                          4625          38.5  Monitor only -- no action recommended from thi...
ctr_only_underperformer          3347          27.8  Rewrite title/meta description -- position and...
dual_underperformer              2541          21.1  Full review, highest priority -- both entry (C...
engagement_only_underperformer   1510          12.6  Review on-page content/layout -- clicks are fi...


**The decay/refresh insight.** Week 4's signal audit already found position drifts modestly worse with content age (7.9 → 8.4 → 8.6 median position). Looking at it through the archetype lens sharpens that into something more actionable: the `dual_underperformer` rate itself climbs with age, not just raw position.

In [3]:
# content_age_days now travels WITH `ranked` (added to `extra` before the sort), so this is
# correctly row-aligned -- unlike attaching it positionally from `lane4` after `ranked` has
# already been reordered by pred_prob, which silently scrambles the mapping.
ranked['age_bucket'] = pd.cut(ranked['content_age_days'], bins=[0, 180, 365, 730, 100000],
                               labels=['<6mo', '6-12mo', '1-2yr', '2yr+'])
decay = ranked.groupby('age_bucket', observed=True).agg(
    n=('archetype', 'size'),
    median_position=('avg_position', 'median'),
    pct_dual_underperformer=('archetype', lambda s: (s == 'dual_underperformer').mean())
)
print(decay.round(3))
print()
rates = decay['pct_dual_underperformer']
print(f"The dual-underperformer rate moves from {rates.iloc[0]*100:.1f}% (<6mo) to "
      f"{rates.iloc[1]*100:.1f}% (6-12mo) to {rates.iloc[2]*100:.1f}% (1-2yr), all buckets well "
      f"above the ~50-row floor.")
if rates.iloc[-1] > rates.iloc[0]:
    print('This IS a real rising trend -- pages crossing the 1-year mark are more likely to need')
    print('both a title/meta AND a content review, not just one or the other.')
else:
    print('The rate is roughly FLAT across age bands in this slice -- age alone does not predict')
    print('needing BOTH fixes at once as cleanly as raw position drift (Week 4) suggested. Worth')
    print('reporting honestly rather than forcing a decay story the archetype data does not show here.')

               n  median_position  pct_dual_underperformer
age_bucket                                                
<6mo        5252              7.9                    0.177
6-12mo      3992              8.4                    0.205
1-2yr       2779              8.6                    0.286

The dual-underperformer rate moves from 17.7% (<6mo) to 20.5% (6-12mo) to 28.6% (1-2yr), all buckets well above the ~50-row floor.
This IS a real rising trend -- pages crossing the 1-year mark are more likely to need
both a title/meta AND a content review, not just one or the other.


**Cost/value thinking.** Not every archetype is worth the same reviewer effort. Checking average impression volume per archetype shows where the cheapest fixes sit on the highest-value real estate.

In [4]:
value_table = ranked.groupby('archetype')['impressions_90d'].agg(['count', 'median', 'mean']).round(0)
print(value_table)
print()
print('ctr_only_underperformer sits on the highest-traffic pages on average (mean 11,915 impressions) --')
print('and a title/meta rewrite is a cheap, fast fix. dual_underperformer and engagement_only pages carry')
print('much lower traffic (median under 2,000) and need a heavier content review. Read together with the')
print('decay insight above: prioritizing ctr_only fixes on older, high-traffic pages is likely the best')
print('reviewer-time-to-value ratio in this queue -- cheap action, real audience, and rising need with age.')

                                count  median     mean
archetype                                             
ctr_only_underperformer          3347  3693.0  11915.0
dual_underperformer              2541  1981.0   4580.0
engagement_only_underperformer   1510  1870.0   3653.0
healthy                          4625  5407.0  13353.0

ctr_only_underperformer sits on the highest-traffic pages on average (mean 11,915 impressions) --
and a title/meta rewrite is a cheap, fast fix. dual_underperformer and engagement_only pages carry
much lower traffic (median under 2,000) and need a heavier content review. Read together with the
decay insight above: prioritizing ctr_only fixes on older, high-traffic pages is likely the best
reviewer-time-to-value ratio in this queue -- cheap action, real audience, and rising need with age.


## 2. Intended use and limits

**Who uses this:** a FlyRank content/SEO reviewer deciding which of Lane 4's visible pages to open first in a given review cycle, when review capacity is limited.

**What it's for:** prioritizing *attention*, not automating a decision. The queue and archetype mapping say "look at this one before that one, and here's roughly what kind of fix it needs" — not "definitely rewrite this."

**Where it stops being valid:**
- **Sample scope** — a 30,000-row anonymized starter slice across 28 clients. Week 6 showed a naive split can inflate precision@20 from an honest 0.600 to an optimistic 0.750; this has not been validated at full warehouse scale (~79M rows, ~70 clients).
- **Same-window label** — both `ctr_gap_score` and `engagement_deficit` are computed from the *current* 90-day window, not a validated future outcome. This queue tells you who looks troubled **right now**.
- **No causal claim** — nothing here says a specific fix will help. That needs an actual before/after experiment, which this data can't provide.

In [5]:
# No additional query needed here -- the scope/limits above are qualitative, grounded in
# numbers already established in Weeks 1, 5, and 6.

## 3. Human review + the no-go list

**What a person must check before acting on any ranked page:**
- Is the low CTR/engagement a real content problem, or a tracking artifact (GSC/GA4 mismatch, a recent URL change, a redirect)? Week 4's top-20 review already found one likely example of this.
- Does the page actually belong to this client/site as expected, or is it an edge case (thin page, wrong intent classification, recently migrated)?
- Is the page part of a larger set (a paginated series, a template family) where "fixing" this one instance in isolation doesn't make sense?

**What should never be automated from this queue alone:**
- Auto-publishing a rewritten title/meta description without human review — the model has no signal about whether a specific rewrite is good, only that current numbers look weak.
- Auto-deleting, merging, or de-indexing any page based on this score — higher-stakes, harder-to-reverse actions than a content review, never validated against that kind of decision.
- Treating `high_confidence` or any archetype label as "certainly wrong" — it means "enough volume and score to trust the ranking," not "guaranteed to need a fix."

In [6]:
# No additional query needed here -- this section states policy/process guardrails,
# not a data claim that needs its own verification query.

## 4. Monitoring / retrain triggers

Storing the current reference numbers so a future run can compare against them — committed as `work/outputs/metrics.json`, since a drift check only means something with a real number to check against.

In [7]:
reference_metrics = {
    'label_base_rate': float(lane4_enc['engagement_deficit'].mean()),
    'high_confidence_count': int(ranked['high_confidence'].sum()),
    'high_confidence_pct': float(ranked['high_confidence'].mean()),
    'archetype_pct': ranked['archetype'].value_counts(normalize=True).round(4).to_dict(),
    'dual_underperformer_rate_by_age': decay['pct_dual_underperformer'].round(4).to_dict(),
    'model_precision_at_20_honest_holdout': 0.600,
    'model_precision_at_50_honest_holdout': 0.520,
    'baseline_precision_at_20_honest_holdout': 0.350,
}
for k, v in reference_metrics.items():
    print(f'{k}: {v}')

label_base_rate: 0.33693753638858853
high_confidence_count: 2405
high_confidence_pct: 0.2000332695666639
archetype_pct: {'healthy': 0.3847, 'ctr_only_underperformer': 0.2784, 'dual_underperformer': 0.2113, 'engagement_only_underperformer': 0.1256}
dual_underperformer_rate_by_age: {'<6mo': 0.1767, '6-12mo': 0.2049, '1-2yr': 0.2861}
model_precision_at_20_honest_holdout: 0.6
model_precision_at_50_honest_holdout: 0.52
baseline_precision_at_20_honest_holdout: 0.35


**Retrain/re-review triggers, concretely:**
- **Label base rate drifts** — if `engagement_deficit`'s rate moves meaningfully away from 0.337, calibration assumptions no longer match reality.
- **Archetype mix shifts** — if `dual_underperformer` share moves far from ~21%, or the age-vs-archetype relationship flattens/reverses, the underlying content mix has changed enough to re-check the whole approach, not just re-run the same thresholds.
- **`high_confidence` share drifts far from ~20%** — the score distribution has likely shifted; recalculate the threshold rather than reapply the old one.
- **New warehouse data becomes available** — this is built entirely from the starter slice; moving to the full warehouse release is itself a retrain trigger.
- **A full quarter passes** — age, freshness, and seasonal patterns all shift with time; a snapshot-based queue degrades in ways this data alone can't detect.

In [8]:
# No additional query needed here -- the triggers above are qualitative policy, built on
# the reference numbers already printed and stored in the previous cell.

## 5. Exports for the paper

Per this week's card: the ranked queue CSV is regenerated by this notebook and stays **out of git** (`work/**/*.csv` is gitignored in this repo — confirmed by checking `.gitignore` directly, not assumed). Figures go to `work/figures/` and get committed. The metrics JSON also gets committed — it's the receipt the paper's numbers trace back to.

In [9]:
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# Queue CSV -- regenerated each run, intentionally NOT committed (gitignored: work/**/*.csv)
export_cols = ['rank', 'content_id', 'client_id', 'position_tier', 'impressions_90d', 'ctr',
               'pred_prob', 'ctr_gap_score', 'archetype', 'reason_low_ctr_vs_tier',
               'reason_high_stakes', 'reason_model_flagged', 'high_confidence']
ranked[export_cols].to_csv('../outputs/action_playbook.csv', index=False)
print(f'wrote {len(ranked)} rows to work/outputs/action_playbook.csv (gitignored by design)')

# Metrics JSON -- committed, the receipts the paper's numbers trace back to
with open('../outputs/metrics.json', 'w') as f:
    json.dump(reference_metrics, f, indent=2)
print('wrote work/outputs/metrics.json (committed)')

# Figure -- committed to work/figures/, per this week's card
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

labels = ['precision@20', 'precision@50']
baseline_vals = [0.350, 0.360]
model_vals = [0.600, 0.520]
x = range(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([i - width/2 for i in x], baseline_vals, width, label='Baseline (ctr_gap rule)', color='#999999')
ax.bar([i + width/2 for i in x], model_vals, width, label='Model (Random Forest)', color='#2e7d6e')
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('Precision')
ax.set_title('Baseline vs Model, honest client-grouped test set')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../figures/baseline_vs_model_precision.png', dpi=150)
plt.close()
print('wrote work/figures/baseline_vs_model_precision.png (committed)')

# Second figure: archetype x age decay chart, also worth reusing in the paper
fig, ax = plt.subplots(figsize=(6, 4))
decay['pct_dual_underperformer'].plot(kind='bar', ax=ax, color='#c0533e')
ax.set_ylabel('% dual_underperformer')
ax.set_xlabel('Content age')
ax.set_title('Dual-underperformer rate rises with content age')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../figures/dual_underperformer_by_age.png', dpi=150)
plt.close()
print('wrote work/figures/dual_underperformer_by_age.png (committed)')

wrote 12023 rows to work/outputs/action_playbook.csv (gitignored by design)
wrote work/outputs/metrics.json (committed)
wrote work/figures/baseline_vs_model_precision.png (committed)
wrote work/figures/dual_underperformer_by_age.png (committed)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.